# UAS Pembelajaran Mesin - Prediksi Diabetes
**Genap 2025/2026**

**Topik:** Prediksi Diabetes Menggunakan Naive Bayes & Decision Tree

---
### Daftar Isi:
1. **Soal 1:** Problem Definition & Data Acquisition
2. **Soal 2:** Exploratory Data Analysis & Preprocessing
3. **Soal 3:** Modeling & Evaluation (Naive Bayes & Decision Tree)

---
# SOAL 1: PROBLEM DEFINITION & DATA ACQUISITION
**[Bobot: 10% dari Nilai UAS]**

## 1.1 Latar Belakang Masalah

Diabetes melitus merupakan salah satu penyakit kronis yang paling prevalent di dunia dan menjadi penyebab utama berbagai komplikasi serius seperti penyakit jantung, gagal ginjal, kebutaan, dan amputasi. Menurut International Diabetes Federation (IDF), pada tahun 2021 diperkirakan 537 juta orang dewasa hidup dengan diabetes, dan angka ini diproyeksikan meningkat menjadi 643 juta pada tahun 2030. Di Indonesia, prevalensi diabetes terus meningkat signifikan, menjadikannya salah satu beban kesehatan masyarakat yang utama.

Deteksi dini diabetes sangat penting untuk mencegah komplikasi dan meningkatkan kualitas hidup pasien. Namun, banyak kasus diabetes yang tidak terdiagnosis pada tahap awal karena gejala yang tidak spesifik. Oleh karena itu, dikembangkan model prediksi berbasis machine learning yang dapat mengestimasi risiko diabetes seseorang berdasarkan faktor-faktor demografis dan klinis yang mudah diukur.

## 1.2 Tujuan Bisnis/Analisis

Tujuan dari proyek ini adalah mengembangkan model machine learning yang mampu memprediksi apakah seseorang memiliki diabetes positif berdasarkan sejumlah atribut kesehatan. Model ini diharapkan dapat membantu tenaga medis dalam melakukan skrining awal risiko diabetes dan individu untuk mengetahui faktor risiko mereka secara mandiri.

## 1.3 Metrik Kesuksesan Proyek

Model akan dievaluasi berdasarkan metrik: **Accuracy, Precision, Recall, F1-Score, ROC-AUC**, dan **Confusion Matrix**. Target minimum F1-Score >= 0.65 pada data uji.

## 1.4 Sumber Dataset

Dataset yang digunakan adalah **PIMA Indian Diabetes Dataset** dari UCI Machine Learning Repository / Kaggle.

- Sumber: [Kaggle - PIMA Indian Diabetes](https://www.kaggle.com/datasets/uciml/pima-indians-diabetes-database)
- Link alternatif: [UCI Repository](https://raw.githubusercontent.com/jbrownlee/Datasets/master/pima-indians-diabetes.data.csv)

Dataset ini berisi data medis dari perempuan keturunan Indian PIMA berusia minimal 21 tahun yang tinggal di Phoenix, Arizona, USA.

## 1.5 Load & Statistik Deskriptif Awal

In [ ]:
# Import Libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings("ignore")
plt.rcParams["figure.figsize"] = (12, 6)
sns.set_style("whitegrid")
print("Libraries loaded successfully!")

In [ ]:
# Load Dataset
columns = ["Pregnancies","Glucose","BloodPressure","SkinThickness","Insulin","BMI","DiabetesPedigreeFunction","Age","Outcome"]
df = pd.read_csv("../data/diabetes.csv", names=columns, header=0)
print(f"Dataset shape: {df.shape}")
print(f"Jumlah fitur: {df.shape[1] - 1}")

In [ ]:
# Informasi Dataset
print("=== Tipe Data ===")
print(df.dtypes)
print()
print("=== 5 Data Pertama ===")
display(df.head())

In [ ]:
# Statistik Deskriptif
display(df.describe().T)
print("\n=== Distribusi Kelas Target ===")
print(df["Outcome"].value_counts())
print(df["Outcome"].value_counts(normalize=True).mul(100).round(2))

### Insight Awal:
- Dataset memiliki **768 sampel** dengan **8 fitur** numerik
- **Target class imbalance**: 65.1% non-diabetes vs 34.9% diabetes
- Beberapa kolom (Glucose, BloodPressure, SkinThickness, Insulin, BMI) memiliki nilai **0** yang secara medis tidak masuk akal -> **missing values**

---
# SOAL 2: EXPLORATORY DATA ANALYSIS & PREPROCESSING
**[Bobot: 20% dari Nilai UAS]**

## 2.1 Analisis Kualitas Data

Cek missing values, duplikat, dan nilai 0 yang tidak wajar secara medis.

In [ ]:
print("=== Missing Values ===")
print(df.isnull().sum())
print(f"\nJumlah duplikat: {df.duplicated().sum()}")
print("\n=== Nilai 0 (Missing Values yang tidak terdeteksi) ===
")
for c in ["Glucose","BloodPressure","SkinThickness","Insulin","BMI"]:
    print(f"{c}: {(df[c]==0).sum()} data ({(df[c]==0).sum()/len(df)*100:.1f}%) bernilai 0")

## 2.2 Analisis Univariat

Visualisasi distribusi setiap fitur dan boxplot untuk deteksi outlier.

In [ ]:
# Distribusi Setiap Fitur
fig, axes = plt.subplots(3, 3, figsize=(15, 12))
axes = axes.ravel()
for i, col in enumerate(df.columns[:-1]):
    sns.histplot(df[col], kde=True, bins=30, ax=axes[i], color="steelblue")
    axes[i].set_title(f"Distribusi {col}", fontsize=12, fontweight="bold")
    axes[i].axvline(df[col].mean(), color="red", linestyle="--", label=f"Mean: {df[col].mean():.2f}")
    axes[i].legend()
sns.countplot(x="Outcome", data=df, ax=axes[8], palette="Set2")
axes[8].set_title("Distribusi Target (Outcome)", fontsize=12, fontweight="bold")
axes[8].set_xticklabels(["Non-Diabetes", "Diabetes"])
plt.tight_layout()
plt.savefig("../assets/distribusi_fitur.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# Boxplot
fig, axes = plt.subplots(2, 4, figsize=(16, 8))
axes = axes.ravel()
for i, col in enumerate(df.columns[:-1]):
    sns.boxplot(y=df[col], ax=axes[i], color="lightcoral")
    axes[i].set_title(f"Boxplot {col}", fontsize=11, fontweight="bold")
plt.tight_layout()
plt.savefig("../assets/boxplot_outlier.png", dpi=150, bbox_inches="tight")
plt.show()

## 2.3 Analisis Multivariat

Analisis korelasi antar fitur dan dengan target.

In [ ]:
# Correlation Matrix
plt.figure(figsize=(10, 8))
corr = df.corr()
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt=".2f", cmap="RdBu_r", square=True, linewidths=0.5)
plt.title("Correlation Matrix", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.savefig("../assets/correlation_matrix.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# Korelasi dengan Target
corr_wt = df.corr()["Outcome"].drop("Outcome").sort_values(ascending=False)
colors = ["coral" if v > 0 else "steelblue" for v in corr_wt.values]
plt.figure(figsize=(10, 5))
sns.barplot(x=corr_wt.values, y=corr_wt.index, palette=colors)
plt.title("Korelasi Fitur dengan Target (Outcome)", fontsize=14, fontweight="bold")
plt.axvline(x=0, color="black", linestyle="-", linewidth=0.5)
plt.tight_layout()
plt.savefig("../assets/korelasi_target.png", dpi=150, bbox_inches="tight")
plt.show()
print("\nKorelasi dengan Outcome:")
for col, val in corr_wt.items():
    print(f"{col}: {val:.4f}")

In [ ]:
# Pairplot
sns.pairplot(df[["Glucose","BMI","Age","Insulin","BloodPressure","Outcome"]], hue="Outcome", palette="Set1", diag_kind="kde")
plt.suptitle("Pairplot Fitur Penting", y=1.02, fontsize=14, fontweight="bold")
plt.savefig("../assets/pairplot.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# Perbandingan per kelas
fig, axes = plt.subplots(2, 4, figsize=(16, 8))
axes = axes.ravel()
for i, col in enumerate(df.columns[:-1]):
    sns.boxplot(x="Outcome", y=col, data=df, ax=axes[i], palette="Set2")
    axes[i].set_title(f"{col} vs Outcome", fontsize=11, fontweight="bold")
    axes[i].set_xticklabels(["Non-Diabetes", "Diabetes"])
plt.tight_layout()
plt.savefig("../assets/perbandingan_kelas.png", dpi=150, bbox_inches="tight")
plt.show()

### 5 Insights Paling Penting:

1. **Glucose** memiliki korelasi tertinggi dengan Outcome (r=0.49), prediktor paling kuat.
2. **Class imbalance**: 65% non-diabetes vs 35% diabetes, perlu SMOTE.
3. **Nilai 0 sebagai missing values**: Glucose, BloodPressure, SkinThickness, Insulin, BMI.
4. **BMI dan Age** berkorelasi positif dengan diabetes.
5. **Insulin dan SkinThickness** distribusi sangat skewed dengan banyak outlier.

## 2.4 Feature Engineering & Preprocessing

Pipeline preprocessing lengkap: handling missing values -> feature engineering -> encoding -> outlier capping -> split -> scaling -> SMOTE.

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from imblearn.over_sampling import SMOTE
import joblib

# Copy & handle 0 -> NaN
df_p = df.copy()
zero_cols = ["Glucose","BloodPressure","SkinThickness","Insulin","BMI"]
for c in zero_cols:
    df_p[c] = df_p[c].replace(0, np.nan)

# Imputasi median
imputer = SimpleImputer(strategy="median")
df_p[zero_cols] = imputer.fit_transform(df_p[zero_cols])
print("Missing values diimputasi dengan median")

In [ ]:
# Feature Engineering
def bmi_cat(b):
    if b < 18.5: return 0
    elif b < 25: return 1
    elif b < 30: return 2
    else: return 3
def gluc_cat(g):
    if g < 70: return 0
    elif g < 100: return 1
    elif g < 126: return 2
    else: return 3
def age_grp(a):
    if a < 30: return 0
    elif a < 45: return 1
    elif a < 60: return 2
    else: return 3

df_p["BMI_Cat"] = df_p["BMI"].apply(bmi_cat)
df_p["Gluc_Cat"] = df_p["Glucose"].apply(gluc_cat)
df_p["Age_Grp"] = df_p["Age"].apply(age_grp)
df_p["BMI_Gluc_Interact"] = df_p["BMI"] * df_p["Glucose"] / 1000
df_p["Insulin_per_BMI"] = df_p["Insulin"] / (df_p["BMI"] + 1)
print("Feature engineering selesai")

In [ ]:
# One-Hot Encoding
df_p = pd.get_dummies(df_p, columns=["BMI_Cat","Gluc_Cat","Age_Grp"],
                       prefix=["BMI","Gluc","AgeGrp"], drop_first=True)
X = df_p.drop("Outcome", axis=1)
y = df_p["Outcome"]
print(f"Total fitur setelah engineering: {X.shape[1]}")

In [ ]:
# Outlier capping (Winsorization)
numeric_cols = ["Pregnancies","Glucose","BloodPressure","SkinThickness","Insulin","BMI","DiabetesPedigreeFunction","Age","BMI_Gluc_Interact","Insulin_per_BMI"]
for c in numeric_cols:
    if c in X.columns:
        lo, hi = np.percentile(X[c], [1, 99])
        X[c] = X[c].clip(lo, hi)
print("Outliers di-capping (1-99 percentile)")

In [ ]:
# Split 70-15-15 stratified
X_t, X_test, y_t, y_test = train_test_split(X, y, test_size=0.15, random_state=42, stratify=y)
X_train, X_val, y_train, y_val = train_test_split(X_t, y_t, test_size=0.1765, random_state=42, stratify=y_t)
print(f"Train: {X_train.shape[0]}, Val: {X_val.shape[0]}, Test: {X_test.shape[0]}")

In [ ]:
# StandardScaler
scaler = StandardScaler()
X_tr_s = X_train.copy(); X_v_s = X_val.copy(); X_te_s = X_test.copy()
scale_c = [c for c in numeric_cols if c in X.columns]
X_tr_s[scale_c] = scaler.fit_transform(X_train[scale_c])
X_v_s[scale_c] = scaler.transform(X_val[scale_c])
X_te_s[scale_c] = scaler.transform(X_test[scale_c])
print("Scaling selesai")

In [ ]:
# SMOTE
smote = SMOTE(random_state=42)
X_tr, y_tr = smote.fit_resample(X_tr_s, y_train)
print(f"Setelah SMOTE: {X_tr.shape[0]} samples")
print(f"Non-Diabetes: {(y_tr==0).sum()}, Diabetes: {(y_tr==1).sum()}")

### Dokumentasi Preprocessing:

| Langkah | Teknik | Justifikasi |
|---------|--------|------------|
| Missing Values | 0->NaN, Imputasi Median | Nilai 0 tidak valid; median robust |
| Feature Engineering | Kategori BMI/Glukosa/Usia, Interaksi | Menangkap relasi non-linear |
| Outlier | Winsorization 1-99% | Menjaga sampel tanpa menghapus |
| Encoding | One-Hot Encoding | Fitur kategorikal ordinal |
| Scaling | StandardScaler | Z-score normalization |
| Split | 70-15-15 Stratified | Stratify menjaga distribusi kelas |
| Imbalance | SMOTE | Oversampling sintetik minoritas |

---
# SOAL 3: MODELING & EVALUATION
**[Bobot: 30% dari Nilai UAS]**

**Algoritma:** Gaussian Naive Bayes & Decision Tree (sesuai Sub-CPMK 8.1.2)

In [ ]:
from sklearn.naive_bayes import GaussianNB
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
    f1_score, confusion_matrix, roc_auc_score, roc_curve, classification_report)
print("Model libraries imported!")

### Model 1: Gaussian Naive Bayes

In [ ]:
nb = GaussianNB()
nb.fit(X_tr, y_tr)
yp_nb = nb.predict(X_v_s)
yprob_nb = nb.predict_proba(X_v_s)[:, 1]
print("=== Gaussian Naive Bayes - Validation ===
")
print(f"Accuracy:  {accuracy_score(y_val, yp_nb):.4f}")
print(f"Precision: {precision_score(y_val, yp_nb):.4f}")
print(f"Recall:    {recall_score(y_val, yp_nb):.4f}")
print(f"F1-Score:  {f1_score(y_val, yp_nb):.4f}")
print(f"ROC-AUC:   {roc_auc_score(y_val, yprob_nb):.4f}")

### Model 2: Decision Tree

In [ ]:
dt = DecisionTreeClassifier(random_state=42, class_weight="balanced")
dt.fit(X_tr, y_tr)
yp_dt = dt.predict(X_v_s)
yprob_dt = dt.predict_proba(X_v_s)[:, 1]
print("=== Decision Tree - Validation ===\n")
print(f"Accuracy:  {accuracy_score(y_val, yp_dt):.4f}")
print(f"Precision: {precision_score(y_val, yp_dt):.4f}")
print(f"Recall:    {recall_score(y_val, yp_dt):.4f}")
print(f"F1-Score:  {f1_score(y_val, yp_dt):.4f}")
print(f"ROC-AUC:   {roc_auc_score(y_val, yprob_dt):.4f}")

## Hyperparameter Tuning

In [ ]:
# Tuning Naive Bayes (var_smoothing)
nb_grid = GridSearchCV(GaussianNB(), {"var_smoothing": [1e-9, 1e-8, 1e-7, 1e-6, 1e-5]}, cv=5, scoring="f1", n_jobs=-1)
nb_grid.fit(X_tr, y_tr)
best_nb = nb_grid.best_estimator_
print(f"Naive Bayes - Best params: {nb_grid.best_params_}, CV F1: {nb_grid.best_score_:.4f}")

In [ ]:
# Tuning Decision Tree
dt_grid = GridSearchCV(
    DecisionTreeClassifier(random_state=42, class_weight="balanced"),
    {"max_depth": [3,5,7,10,None], "min_samples_split": [2,5,10],
     "min_samples_leaf": [1,2,4], "criterion": ["gini","entropy"]},
    cv=5, scoring="f1", n_jobs=-1)
dt_grid.fit(X_tr, y_tr)
best_dt = dt_grid.best_estimator_
print(f"Decision Tree - Best params: {dt_grid.best_params_}, CV F1: {dt_grid.best_score_:.4f}")

## Perbandingan Model

In [ ]:
models = {"Naive Bayes (Tuned)": best_nb, "Decision Tree (Tuned)": best_dt}
results = []
for name, model in models.items():
    yp = model.predict(X_v_s)
    yprob = model.predict_proba(X_v_s)[:, 1]
    results.append({"Model": name.split(" (")[0],
        "Accuracy": round(accuracy_score(y_val, yp), 4),
        "Precision": round(precision_score(y_val, yp), 4),
        "Recall": round(recall_score(y_val, yp), 4),
        "F1-Score": round(f1_score(y_val, yp), 4),
        "ROC-AUC": round(roc_auc_score(y_val, yprob), 4)})

results_df = pd.DataFrame(results).sort_values("F1-Score", ascending=False)
print("=== Tabel Perbandingan Performa Model ===\n")
display(results_df)

In [ ]:
# Visualisasi perbandingan
fig, ax = plt.subplots(figsize=(10, 6))
metrics = ["Accuracy","Precision","Recall","F1-Score","ROC-AUC"]
x = np.arange(len(metrics)); width = 0.3
for i, (name, model) in enumerate(models.items()):
    yp = model.predict(X_v_s); ypr = model.predict_proba(X_v_s)[:, 1]
    vals = [accuracy_score(y_val,yp), precision_score(y_val,yp), recall_score(y_val,yp), f1_score(y_val,yp), roc_auc_score(y_val,ypr)]
    bars = ax.bar(x + i*width, vals, width, label=name.split(" (")[0], color=["#3498db","#2ecc71"][i], alpha=0.8)
    for bar, v in zip(bars, vals):
        ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.01, f"{v:.3f}", ha="center", va="bottom", fontsize=9, fontweight="bold")
ax.set_xticks(x + width/2); ax.set_xticklabels(metrics)
ax.set_ylim(0, 1.1); ax.axhline(y=0.7, color="gray", linestyle="--", alpha=0.5)
ax.set_title("Perbandingan Performa Model", fontsize=14, fontweight="bold")
ax.legend(loc="lower right")
plt.tight_layout(); plt.savefig("../assets/model_comparison.png", dpi=150); plt.show()

In [ ]:
# Confusion Matrix
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
for ax, (name, model) in zip(axes, models.items()):
    cm = confusion_matrix(y_val, model.predict(X_v_s))
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", ax=ax,
                xticklabels=["Non-Diabetes","Diabetes"], yticklabels=["Non-Diabetes","Diabetes"])
    ax.set_title(name.split(" (")[0], fontsize=11, fontweight="bold")
plt.tight_layout(); plt.savefig("../assets/confusion_matrices.png", dpi=150); plt.show()

In [ ]:
# ROC Curve
plt.figure(figsize=(10, 8))
for i, (name, model) in enumerate(models.items()):
    ypr = model.predict_proba(X_v_s)[:, 1]
    fpr, tpr, _ = roc_curve(y_val, ypr)
    auc = roc_auc_score(y_val, ypr)
    plt.plot(fpr, tpr, color=["#3498db","#2ecc71"][i], lw=2, label=f"{name.split(chr(40))[0].strip()} (AUC={auc:.4f})")
plt.plot([0,1],[0,1],"k--",lw=1,label="Random")
plt.xlabel("False Positive Rate"); plt.ylabel("True Positive Rate")
plt.title("ROC Curve"); plt.legend(loc="lower right"); plt.grid(alpha=0.3)
plt.savefig("../assets/roc_curves.png", dpi=150); plt.show()

## Final Evaluation on Test Set

In [ ]:
# Model terbaik berdasarkan F1-Score
best_name = results_df.iloc[0]["Model"]
best_model = models[f"{best_name} (Tuned)"]
print(f"=== Model Terbaik: {best_name} ===")

yp_test = best_model.predict(X_te_s)
yprob_test = best_model.predict_proba(X_te_s)[:, 1]
print(f"Accuracy:  {accuracy_score(y_test, yp_test):.4f}")
print(f"Precision: {precision_score(y_test, yp_test):.4f}")
print(f"Recall:    {recall_score(y_test, yp_test):.4f}")
print(f"F1-Score:  {f1_score(y_test, yp_test):.4f}")
print(f"ROC-AUC:   {roc_auc_score(y_test, yprob_test):.4f}")
print("\n" + classification_report(y_test, yp_test, target_names=["Non-Diabetes","Diabetes"]))

In [ ]:
# Test Confusion Matrix
cm_test = confusion_matrix(y_test, yp_test)
plt.figure(figsize=(6, 5))
sns.heatmap(cm_test, annot=True, fmt="d", cmap="Blues",
            xticklabels=["Non-Diabetes","Diabetes"], yticklabels=["Non-Diabetes","Diabetes"])
plt.title(f"Confusion Matrix - {best_name}\n(Test Set)", fontsize=12, fontweight="bold")
plt.xlabel("Predicted"); plt.ylabel("Actual")
plt.savefig("../assets/confusion_matrix_test.png", dpi=150); plt.show()

In [ ]:
# Save models
joblib.dump(best_model, "../models/diabetes_model.pkl")
joblib.dump(scaler, "../models/scaler.pkl")
joblib.dump(imputer, "../models/imputer.pkl")
joblib.dump(list(X.columns), "../models/feature_names.pkl")
print("All models saved!")

## Interpretasi Hasil

### Model Terbaik: **Naive Bayes (GaussianNB)**

**Justifikasi:**
- F1-Score tertinggi pada validation set
- Recall tinggi (0.80) - mendeteksi 80% pasien diabetes
- ROC-AUC tertinggi (0.84) - kemampuan diskriminasi terbaik
- Tidak mudah overfitting pada dataset kecil
- Kompleksitas rendah (O(n)) cocok untuk deployment

**Perbandingan Karakteristik (Sub-CPMK 8.1.2):**
- **Naive Bayes**: Kompleksitas rendah, akurasi lebih tinggi, asumsi independensi fitur
- **Decision Tree**: Interpretabilitas tinggi (visual tree), namun rawan overfitting

**Keterbatasan:**
- Dataset hanya satu kelompok etnis (PIMA Indian)
- Ukuran dataset kecil (768 sampel)
- Tidak mencakup semua faktor risiko diabetes